In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git" 
REPO_NAME = "tranSTR_Casual"
BRANCH = "groundedDINO" 

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

# Change Directory to the repo root 
if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
             os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
             os.chdir(REPO_NAME)
        
        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
             print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install & Login W&B (với API Key trực tiếp)
print('=== CELL 2: W&B Setup ===')
!pip install -q wandb peft --upgrade
import wandb

# ============================================
# WANDB CONFIG - THAY THẾ BẰNG API KEY CỦA BẠN
# ============================================
WANDB_API_KEY = ''
WANDB_PROJECT = 'transtr-causalvid-gdino-lora-ncod-hum'
WANDB_ENTITY = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in successfully!')

In [ ]:
# CELL 3: Imports
print('=== CELL 3: Imports ===')
import os, torch, numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import torch.nn.functional as F
print('Imports OK')

In [ ]:
# CELL 4: NCOD + HUM + Verifier/Knowledge training functions
print('=== CELL 4: Functions (NCOD + HUM + Verifier/Knowledge) ===')

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=4, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_mask = (q_family_id >= 2)
        l1 = torch.where(hard_mask, hum_loss, lum_loss).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            correct += (logits.argmax(-1) == tgt).sum().item()
            total += tgt.size(0)
    return correct / max(total, 1) * 100

print('Functions defined for integrated training.')

In [ ]:
# CELL 5: Setup Paths & Config
print('=== CELL 5: Paths & Config ===')

CLIP_FEATURE_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = '/kaggle/input/datasets/danielq07/gdino-roi-all-nodes-merged'
ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'
SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'

BASE = '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

RUN_TRAINING = True
MODEL_FILENAME = 'best_model_gdino_lora_ncod_hum.ckpt'
FEAT_DIM = 1024

class Config:
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = 1028
    use_grounding_dino = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    text_encoder_type = 'microsoft/deberta-v3-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    use_lora = True
    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.1
    lora_target_modules = ['query_proj', 'key_proj', 'value_proj']
    lora_lr = 1e-4

    bs = 8
    accumulation_steps = 4
    lr = 1e-5
    epoch = 10
    gpu = 0
    patience = 5
    gamma = 0.1
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.2
    lambda_knowledge = 0.1
    return_family_id = True

    ncod_u_lr = 0.1
    hum_alpha = 1.0

    num_workers = 4
    hard_eval = False

args = Config()
set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Effective BS: {args.bs * args.accumulation_steps}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# CELL 7: Model + Optimizers + NCOD U
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

non_text_params = []
lora_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' not in name:
        non_text_params.append(p)
    elif 'lora_' in name:
        lora_params.append(p)
    else:
        text_base_params.append(p)

param_groups = [
    {'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay},
    {'params': lora_params, 'lr': args.lora_lr, 'weight_decay': args.decay},
]
if len(text_base_params) > 0 and not args.freeze_text_encoder:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(optimizer_model, 'max', factor=args.gamma, patience=args.patience)

U = torch.nn.Parameter(torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device))
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)

xe = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

BEST_ARTIFACT_NAME = 'best-model-gdino-lora-ncod-hum'
LAST_ARTIFACT_NAME = 'last-checkpoint-gdino-lora-ncod-hum'
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'U shape: {tuple(U.shape)}')

In [ ]:
# CELL 8: Init W&B
print('=== CELL 8: Initialize W&B Run ===')
start_epoch = 1
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

wandb_config = {
    'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type,
    'use_lora': args.use_lora,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'lambda_verifier': args.lambda_verifier,
    'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha
}
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url}')

In [ ]:
# CELL 9: Integrated Training Loop
print('=== CELL 9: Training ===')

if RUN_TRAINING:
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')
        total_loss, l1, l2, verifier_loss, knowledge_loss, train_acc = train_epoch_integrated(
            model=model,
            optimizer_model=optimizer_model,
            optimizer_u=optimizer_u,
            U=U,
            loader=train_loader,
            xe=xe,
            bce=bce,
            device=device,
            epoch=ep,
            key_to_idx=train_key_to_idx,
            accumulation_steps=args.accumulation_steps,
            hum_alpha=args.hum_alpha,
            lambda_verifier=args.lambda_verifier,
            lambda_knowledge=args.lambda_knowledge
        )

        val_acc = eval_epoch(model, val_loader, device)
        scheduler.step(val_acc)

        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        wandb.log({
            'epoch': ep,
            'train_total_loss': total_loss,
            'train_l1': l1,
            'train_l2': l2,
            'train_verifier_loss': verifier_loss,
            'train_knowledge_loss': knowledge_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'u_mean': U.detach().mean().item(),
            'u_max': U.detach().max().item()
        })

        ckpt = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_model_state_dict': optimizer_model.state_dict(),
            'optimizer_u_state_dict': optimizer_u.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'history': history,
            'U': U.detach().cpu(),
            'train_sample_keys': train_sample_keys
        }
        torch.save(ckpt, os.path.join(MODEL_DIR, 'latest_checkpoint_gdino_lora_ncod_hum.ckpt'))

        if val_acc > best_acc:
            best_acc = val_acc
            ckpt['best_acc'] = best_acc
            torch.save(ckpt, save_path)
            print(f'New best val_acc={best_acc:.2f}%')

    print(f'Best Val Accuracy: {best_acc:.2f}%')
else:
    print('Skipping training')

In [ ]:
# CELL 10: Detailed Evaluation + CSV export
print('=== CELL 10: Detailed Evaluation ===')
import seaborn as sns

CSV_OUTPUT_PATH = 'test_predictions_gdino_lora_ncod_hum.csv'

def _build_eval_meta_map(loader):
    dataset = getattr(loader, 'dataset', None)
    sample_list = getattr(dataset, 'sample_list', None) if dataset is not None else None
    meta_map = {}
    if sample_list is None:
        return meta_map
    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        qns_key = f'{vid}_{qtype}'
        meta_map[qns_key] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results = {}
    prediction_rows = []
    meta_map = _build_eval_meta_map(loader)

    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            targets = ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                video_id = str(meta.get('video_id', str(key)))
                question = str(meta.get('question', qns[i]))
                answers = meta.get('answers', ['', '', '', '', ''])
                if len(answers) < 5:
                    answers += [''] * (5 - len(answers))
                answers = answers[:5]

                correct_idx = int(target)
                predicted_idx = int(pred)
                is_correct = int(correct_idx == predicted_idx)

                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(is_correct)
                })

                prediction_rows.append({
                    'video_id': video_id,
                    'question_type': qtype,
                    'question': question,
                    'correct_idx': correct_idx,
                    'predicted_idx': predicted_idx,
                    'is_correct': is_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'a0': answers[0],
                    'prob_a0': float(prob_vec[0]),
                    'a1': answers[1],
                    'prob_a1': float(prob_vec[1]),
                    'a2': answers[2],
                    'prob_a2': float(prob_vec[2]),
                    'a3': answers[3],
                    'prob_a3': float(prob_vec[3]),
                    'a4': answers[4],
                    'prob_a4': float(prob_vec[4]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                })

    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)

    metrics = {}
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        total = len(rows)
        correct = sum(1 for r in rows if r['correct'])
        metrics[name] = (correct / total * 100) if total > 0 else 0.0

    if log_to_wandb:
        wandb.log({
            'eval/Description': metrics['Description'],
            'eval/Explanation': metrics['Explanation'],
            'eval/Predictive_Answer': metrics['Predictive-Answer'],
            'eval/Predictive_Reason': metrics['Predictive-Reason'],
            'eval/Counterfactual_Answer': metrics['Counterfactual-Answer'],
            'eval/Counterfactual_Reason': metrics['Counterfactual-Reason']
        })

    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)
print(metrics)

In [ ]:
# CELL 11: Finish W&B Run
print('=== CELL 11: Finish W&B ===')

with open('final_metrics_gdino_lora_ncod_hum.json', 'w') as f:
    json.dump(metrics, f, indent=2)

final_artifact = wandb.Artifact(
    name='final-results-gdino-lora-ncod-hum',
    type='results',
    metadata={
        'backbone': 'dinov3+groundingdino',
        'text_encoder': args.text_encoder_type,
        'lora': args.use_lora,
        'ncod_hum': True
    }
)
final_artifact.add_file('final_metrics_gdino_lora_ncod_hum.json')
if os.path.exists(predictions_csv):
    final_artifact.add_file(predictions_csv)
if os.path.exists(save_path):
    final_artifact.add_file(save_path)
wandb.log_artifact(final_artifact)
wandb.finish()
print('✅ W&B run finished!')